In [ ]:
import os
import time
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from ultralytics import YOLO

from torchvision import models
from torchvision import transforms

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)

from openpyxl import Workbook

##########################################################################
# CONFIGURACIÓN
##########################################################################

DATASET_PATH = "dataset"
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DOG_CLASS_RANGE = range(151, 269)

device = "cuda" if torch.cuda.is_available() else "cpu"

##########################################################################
# MODELOS
##########################################################################

yolo = YOLO("yolov8n.pt")

cnn = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

vit = models.vit_b_16(
    weights=models.ViT_B_16_Weights.IMAGENET1K_V1
)

cnn.to(device)
vit.to(device)

cnn.eval()
vit.eval()

##########################################################################
# TRANSFORMACIONES
##########################################################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

##########################################################################
# UTILIDADES
##########################################################################

def classify_crop(model, image):

    tensor = transform(image)
    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():

        out = model(tensor)

        probs = torch.softmax(out, dim=1)

        dog_prob = probs[
            0,
            list(DOG_CLASS_RANGE)
        ].sum()

    return float(dog_prob)


##########################################################################
# YOLO DETECCIÓN
##########################################################################

def yolo_probability(image_path):

    results = yolo.predict(
        image_path,
        conf=0.05,
        verbose=False
    )

    max_conf = 0

    for result in results:

        for box in result.boxes:

            cls_id = int(box.cls.item())

            if yolo.names[cls_id] == "dog":

                conf = float(box.conf.item())

                max_conf = max(max_conf, conf)

    return max_conf


##########################################################################
# ENSEMBLE
##########################################################################

def ensemble_probability(
        yolo_prob,
        cnn_prob,
        vit_prob,
        weights):

    w1, w2, w3 = weights

    return (
        w1*yolo_prob +
        w2*cnn_prob +
        w3*vit_prob
    )


##########################################################################
# DATASET
##########################################################################

y_true = []

probs_yolo = []
probs_cnn = []
probs_vit = []
probs_ensemble = []

times_yolo = []
times_cnn = []
times_vit = []

##########################################################################
# RECORRIDO
##########################################################################

for category in ["dog", "no_dog"]:

    class_path = os.path.join(
        DATASET_PATH,
        category
    )

    if category == "dog":
        label = 1
    else:
        label = 0

    for file in os.listdir(class_path):

        if not file.lower().endswith(
            (".jpg",".jpeg",".png")
        ):
            continue

        image_path = os.path.join(
            class_path,
            file
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        y_true.append(label)

        ##################################################
        # YOLO
        ##################################################

        start = time.perf_counter()

        yolo_prob = yolo_probability(
            image_path
        )

        times_yolo.append(
            time.perf_counter() - start
        )

        ##################################################
        # CNN
        ##################################################

        start = time.perf_counter()

        cnn_prob = classify_crop(
            cnn,
            image
        )

        times_cnn.append(
            time.perf_counter() - start
        )

        ##################################################
        # ViT
        ##################################################

        start = time.perf_counter()

        vit_prob = classify_crop(
            vit,
            image
        )

        times_vit.append(
            time.perf_counter() - start
        )

        probs_yolo.append(yolo_prob)
        probs_cnn.append(cnn_prob)
        probs_vit.append(vit_prob)

##########################################################################
# PESOS ADAPTATIVOS
##########################################################################

auc_yolo = roc_auc_score(
    y_true,
    probs_yolo
)

auc_cnn = roc_auc_score(
    y_true,
    probs_cnn
)

auc_vit = roc_auc_score(
    y_true,
    probs_vit
)

s = (
    auc_yolo +
    auc_cnn +
    auc_vit
)

weights = (
    auc_yolo/s,
    auc_cnn/s,
    auc_vit/s
)

##########################################################################
# ENSEMBLE
##########################################################################

for yp, cp, vp in zip(
    probs_yolo,
    probs_cnn,
    probs_vit
):

    probs_ensemble.append(

        ensemble_probability(
            yp,
            cp,
            vp,
            weights
        )
    )

##########################################################################
# MÉTRICAS
##########################################################################

def compute_metrics(name,
                    y_true,
                    scores):

    preds = [
        1 if x >= 0.5 else 0
        for x in scores
    ]

    cm = confusion_matrix(
        y_true,
        preds
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {

        "Modelo": name,

        "Accuracy":
        accuracy_score(
            y_true,
            preds
        ),

        "Precision":
        precision_score(
            y_true,
            preds
        ),

        "Recall":
        recall_score(
            y_true,
            preds
        ),

        "F1":
        f1_score(
            y_true,
            preds
        ),

        "ROC_AUC":
        roc_auc_score(
            y_true,
            scores
        ),

        "Specificity":
        tn/(tn+fp),

        "Sensitivity":
        tp/(tp+fn),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return metrics, cm


##########################################################################
# EVALUACIÓN
##########################################################################

m_yolo, cm_yolo = compute_metrics(
    "YOLO",
    y_true,
    probs_yolo
)

m_cnn, cm_cnn = compute_metrics(
    "ResNet50",
    y_true,
    probs_cnn
)

m_vit, cm_vit = compute_metrics(
    "ViT",
    y_true,
    probs_vit
)

m_ens, cm_ens = compute_metrics(
    "ENSEMBLE",
    y_true,
    probs_ensemble
)

##########################################################################
# DATAFRAME
##########################################################################

df = pd.DataFrame([

    m_yolo,
    m_cnn,
    m_vit,
    m_ens

])

##########################################################################
# TIEMPOS
##########################################################################

df["Inference_ms"] = [

    np.mean(times_yolo)*1000,
    np.mean(times_cnn)*1000,
    np.mean(times_vit)*1000,
    (
        np.mean(times_yolo)+
        np.mean(times_cnn)+
        np.mean(times_vit)
    )*1000

]

##########################################################################
# GUARDAR EXCEL
##########################################################################

excel_file = os.path.join(
    OUTPUT_DIR,
    "metricas.xlsx"
)

with pd.ExcelWriter(excel_file) as writer:

    df.to_excel(
        writer,
        sheet_name="Metricas",
        index=False
    )

##########################################################################
# ROC CURVES
##########################################################################

plt.figure(figsize=(8,6))

for name, probs in [

    ("YOLO", probs_yolo),
    ("ResNet50", probs_cnn),
    ("ViT", probs_vit),
    ("Ensemble", probs_ensemble)

]:

    fpr, tpr, _ = roc_curve(
        y_true,
        probs
    )

    roc_auc = auc(
        fpr,
        tpr
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} AUC={roc_auc:.3f}"
    )

plt.plot(
    [0,1],
    [0,1],
    "--"
)

plt.legend()
plt.title("ROC Curve")

plt.savefig(
    f"{OUTPUT_DIR}/roc_all.png"
)

##########################################################################
# PRECISION RECALL
##########################################################################

plt.figure(figsize=(8,6))

for name, probs in [

    ("YOLO", probs_yolo),
    ("ResNet50", probs_cnn),
    ("ViT", probs_vit),
    ("Ensemble", probs_ensemble)

]:

    p,r,_ = precision_recall_curve(
        y_true,
        probs
    )

    plt.plot(
        r,
        p,
        label=name
    )

plt.legend()

plt.title(
    "Precision Recall Curves"
)

plt.savefig(
    f"{OUTPUT_DIR}/pr_all.png"
)

##########################################################################
# COMPARATIVA
##########################################################################

plt.figure(figsize=(12,6))

metrics_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

x = np.arange(
    len(metrics_plot)
)

width = 0.2

for i,row in enumerate(
    df.itertuples()
):

    values = [
        getattr(row,m)
        for m in metrics_plot
    ]

    plt.bar(
        x+i*width,
        values,
        width,
        label=row.Modelo
    )

plt.xticks(
    x + width,
    metrics_plot
)

plt.legend()

plt.savefig(
    f"{OUTPUT_DIR}/comparativa_metricas.png"
)

##########################################################################
# MOSTRAR RESULTADOS
##########################################################################

print("\n")
print("="*80)
print("RESULTADOS FINALES")
print("="*80)

print(df)

print("\nPesos Ensemble:")
print(f"YOLO     : {weights.4f}")
print(f"ResNet50 : {weights.4f}")
print(f"ViT      : {weights.4f}")

print("\nArchivos generados en:")
print(OUTPUT_DIR)

Dado un conjunto de imágenes con animales para clasificación determine si en la misma hay un perro o no. Se usan dos modelos inteligentes preentregandos distintos.  Una CNN y un ViT:
- CNN: ResNet50
- Vision Transformer (ViT): ViT-B/16  
El código agrega un modelo de detección como YOLOv8 junto con ResNet50 y ViT. Esto permite no solo saber si existe un perro, sino también localizarlo mediante un bounding box y comparar CNN vs ViT sobre la misma región de interés, obteniendo métricas más robustas (accuracy, precision, recall y F1-score).